# External Evaluation — DeepSense 6G Scenario 7 — MobileNet+LeNet baseline vs. VIBE

Companion code for:

> **Look Once, Beam Twice: Camera-Primed Real-Time Double-Directional mmWave
> Beam Management for Vehicular Connectivity**
> Avhishek Biswas\*, Apala Pramanik\*, Eylem Ekici, Mehmet C. Vuran (\*equal contribution)
> *Proc. IEEE SECON 2026*, Pisa, Italy.
> Paper (arXiv): <https://arxiv.org/pdf/2605.05071>

On **DeepSense 6G Scenario 7**, this notebook evaluates three beam-selection
methods on the same images and ground-truth mmWave power vectors:

1. **MobileNet+LeNet** — semantic-segmentation + LeNet beam classifier of
   Imran *et al.*, ICC Workshops 2023 (DeepSense ref. [29]).
2. **VIBE-YOLOR** — camera priming (YOLOv11 car detection) + radio-coordinate
   projection only (no closed loop).
3. **VIBE-MA** — VIBE-YOLOR plus the closed-loop moving-average offset
   tracking / neighbour search.

Scenario 7 is a **"seen"** scenario for the MNet+LeNet baseline (it was
trained on it). Results feed Table II / Fig. 11 of the paper.

### You must supply (not shipped in this repo)
- **DeepSense 6G Scenario 7** — https://www.deepsense6g.net/
- **LeNet baseline weights** (`LeNet5_64_beam`) — upstream repo
  https://github.com/convexoptimist/Environment-Semantic-Communication-

Set the data location once in the **CONFIG** cell (or via the
`DEEPSENSE_SCENARIO7_ROOT` environment variable). No other cell needs editing.

In [ ]:
!nvidia-smi

## 1. Imports

In [ ]:
# === Standard library ===
import os
import sys
import time
import logging
from glob import glob, escape

# === Third-party ===
import numpy as np
import pandas as pd
from PIL import Image

# === PyTorch / vision ===
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from ultralytics import YOLO
from transformers import AutoImageProcessor, MobileNetV2ForSemanticSegmentation

## 2. CONFIG — paths & parameters (edit here only)

Everything path/parameter related lives in the next cell. The dataset folder
name contains brackets (`DEV[95]`), so globbing uses `glob.escape`.

In [ ]:
# =============================================================================
# CONFIG  --  edit paths / parameters in THIS cell only
# =============================================================================
# DeepSense 6G Scenario 7. Data you must supply yourself (NOT in this repo):
#   * DeepSense 6G Scenario 7  -> https://www.deepsense6g.net/
#   * LeNet weights LeNet5_64_beam ->
#       https://github.com/convexoptimist/Environment-Semantic-Communication-
#
# Expected layout under SCENARIO7_BASE (note the bracketed folder name):
#   scenario7/DEV[95]/scenario7.csv
#   scenario7/DEV[95]/unit1/camera_data_passes/pass*/*.jpg
#   scenario7/DEV[95]/unit1/mmWave_data/*.txt
#   scenario7/DEV[95]/resources/annotations/bbox/*.txt
#   scenario7/saved_folder/01-29-2023_13_44/checkpoint/LeNet5_64_beam
# =============================================================================
import os
import sys
import pandas as pd

try:
    PROJECT_ROOT = os.path.dirname(os.path.abspath(__file__))
except NameError:
    PROJECT_ROOT = os.getcwd()
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Scenario 7 base dir (contains DEV[95]/ and saved_folder/). Override with:
#   export DEEPSENSE_SCENARIO7_ROOT=/data/DeepSense6G/scenario7
SCENARIO7_BASE = os.environ.get(
    "DEEPSENSE_SCENARIO7_ROOT",
    os.path.join(PROJECT_ROOT, "scenario7"),
)

scenario_root     = os.path.join(SCENARIO7_BASE, "DEV[95]")
scenario_csv_path = os.path.join(scenario_root, "scenario7.csv")
base_dir          = os.path.join(scenario_root, "unit1", "camera_data_passes")
rx_power_base     = os.path.join(scenario_root, "unit1", "mmWave_data")
bbox_base         = os.path.join(scenario_root, "resources", "annotations", "bbox")
bbox_dir          = bbox_base   # alias used by the optional cleanup cell

# LeNet baseline checkpoint (from the upstream Environment-Semantic repo)
LENET_CKPT = os.path.join(
    SCENARIO7_BASE, "saved_folder", "01-29-2023_13_44",
    "checkpoint", "LeNet5_64_beam",
)

# --- Evaluation parameters ---
quantile_percentile = 0.80   # received-power quantile threshold: 0.80 / 0.90 / 0.95
delta               = 10     # VIBE-MA neighbour-search half-window (delta_max)

# --- Output dir for result CSVs ---
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "MNet_LeNet_Model_Analysis")
os.makedirs(OUTPUT_DIR, exist_ok=True)

for _lbl, _p in [("scenario7.csv", scenario_csv_path),
                 ("camera_data_passes", base_dir),
                 ("LeNet checkpoint", LENET_CKPT)]:
    if not os.path.exists(_p):
        print(f"[WARN] {_lbl} not found: {_p}")
print(f"[INFO] PROJECT_ROOT  = {PROJECT_ROOT}")
print(f"[INFO] scenario_root = {scenario_root}")
print(f"[INFO] OUTPUT_DIR    = {OUTPUT_DIR}")

scenario_df = pd.read_csv(scenario_csv_path)
scenario_df["image_name"] = scenario_df["unit1_rgb"].apply(os.path.basename)
image_to_power_path = dict(zip(scenario_df["image_name"], scenario_df["unit1_pwr_60ghz"]))

## 3. (Optional) remove double-vehicle frames

Some Scenario 7 frames have **two** vehicle bounding boxes, which makes the
single-target ground truth ambiguous. The original study removed them. The
next cell can do this, but it **deletes image files from your DeepSense
copy**, so it is guarded by a `DELETE_DOUBLE_VEHICLE` flag (default `False` =
dry-run: it only lists what would be removed).

In [ ]:
# OPTIONAL dataset cleanup -- remove frames that have >1 vehicle bbox.
# SAFETY: this DELETES .jpg files from your DeepSense copy. It is a DRY-RUN by
# default; set DELETE_DOUBLE_VEHICLE = True to actually delete. Paths come from
# the CONFIG cell (base_dir / bbox_dir / scenario_root).
DELETE_DOUBLE_VEHICLE = False   # <-- set True to really delete

removed_images = []
for pass_folder in sorted(os.listdir(base_dir)):
    pass_path = os.path.join(base_dir, pass_folder)
    if not os.path.isdir(pass_path) or not pass_folder.startswith("pass"):
        continue
    print(f"\n[SCAN] {pass_folder}...")
    image_paths = sorted(glob(os.path.join(escape(pass_path), "*.jpg")))
    if not image_paths:
        print(f"[WARN] No images found in {pass_folder}")
        continue
    for image_path in image_paths:
        image_name = os.path.basename(image_path)
        ann_path = os.path.join(bbox_dir, image_name.replace(".jpg", ".txt"))
        if not os.path.exists(ann_path):
            continue
        with open(ann_path, "r") as f:
            lines = [ln for ln in f if ln.strip()]
        if len(lines) > 1:
            removed_images.append(image_name)
            if DELETE_DOUBLE_VEHICLE:
                try:
                    os.remove(image_path)
                    print(f"[DEL ] {image_name} (2 annotations)")
                except Exception as e:
                    print(f"[WARN] could not remove {image_name}: {e}")
            else:
                print(f"[DRY ] would remove {image_name} (2 annotations)")

log_path = os.path.join(scenario_root, "removed_images_with_two_annotations.txt")
if DELETE_DOUBLE_VEHICLE:
    with open(log_path, "w") as f:
        for name in removed_images:
            f.write(name + "\n")
    print(f"\n[INFO] Removed {len(removed_images)} images. Log: {log_path}")
else:
    print(f"\n[INFO] DRY-RUN: {len(removed_images)} images WOULD be removed "
          f"(set DELETE_DOUBLE_VEHICLE=True to delete).")

## 4. MobileNet-V2 — semantic vehicle mask

`generate_mask_from_rgb` runs a DeepLabV3-MobileNetV2 segmentation model and
produces a binary vehicle mask (car/bus/truck). The mask is the input to the
LeNet beam classifier below — this two-stage stack is the MNet+LeNet baseline.

In [ ]:
# -----------------------------------------------------------------------------
# MobileNet-V2 semantic segmentation -> binary vehicle mask (Section 4).
# generate_mask_from_rgb() saves a *_mask.png next to each frame; that mask is
# the input to the LeNet beam classifier. Part of the MNet+LeNet baseline [29].
# -----------------------------------------------------------------------------
# Load pretrained MobileNetV2 segmentation model
image_processor = AutoImageProcessor.from_pretrained("google/deeplabv3_mobilenet_v2_1.0_513")
# model = MobileNetV2ForSemanticSegmentation.from_pretrained("google/deeplabv3_mobilenet_v2_1.0_513").eval()
# Check for GPU availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load model to GPU if available
model = MobileNetV2ForSemanticSegmentation.from_pretrained("google/deeplabv3_mobilenet_v2_1.0_513").to(device).eval()


def generate_mask_from_rgb(image_path, save_path=None, all_cars=True):
    """
    Generate binary mask from RGB image using MobileNetV2-based semantic segmentation.
    If save_path is provided, the binary mask will be saved to that path.

    Parameters:
        image_path (str): path to RGB input image
        save_path (str): path to save generated binary mask
        all_cars (bool): if True, include all vehicle classes; else only 'car'
    
    Returns:
        binary_mask (PIL.Image): generated grayscale mask image
    """
    image = Image.open(image_path).convert("RGB")

    # Preprocess and inference
    # inputs = image_processor(images=image, return_tensors="pt")
    # with torch.no_grad():
    #     outputs = model(**inputs)

    inputs = image_processor(images=image, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits[0]  # shape: [num_classes, H, W]
    predicted_label = logits.argmax(dim=0)  # shape: [H, W]

    # Generate binary mask using COCO class indices
    if all_cars:
        vehicle_classes = [2, 5, 7]  # car, bus, truck
    else:
        vehicle_classes = [2]       # only car

    predicted_np = predicted_label.cpu().numpy()
    binary_mask_np = np.isin(predicted_np, vehicle_classes).astype(np.uint8) * 255

    # Convert to grayscale PIL Image
    binary_mask = Image.fromarray(binary_mask_np).convert("L")

    # Save if requested
    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        binary_mask.save(save_path)

    return binary_mask

## 5. LeNet — mask → beam classifier

`LeNet5` (1-channel 32×32 input, 65-class output) predicts the top-k beam
indices from the MobileNet mask. Checkpoint path comes from the CONFIG cell.

In [ ]:
# -----------------------------------------------------------------------------
# LeNet beam classifier (Section 5) -- Imran et al., ICC Workshops 2023, [29].
# 1-channel 32x32 mask in -> 65-class beam logits; predict() returns top-k.
# Checkpoint path (LENET_CKPT) comes from the CONFIG cell.
# -----------------------------------------------------------------------------
# Constant model checkpoint path
CHECKPOINT_PATH = LENET_CKPT  # from CONFIG
TOP_K = 3
# -------- LeNet Model Definition --------
class LeNet5(nn.Module):
    def __init__(self, num_classes):
        super(LeNet5, self).__init__()
        self.layer1 = nn.Sequential(
            nn.Conv2d(1, 6, kernel_size=5),
            nn.BatchNorm2d(6),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        self.layer2 = nn.Sequential(
            nn.Conv2d(6, 16, kernel_size=5),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        self.fc = nn.Linear(400, 120)
        self.relu = nn.ReLU()
        self.fc1 = nn.Linear(120, 84)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(84, num_classes)

    def forward(self, x):
        x = self.layer1(x)
        x = self.layer2(x)
        x = x.view(x.size(0), -1)
        x = self.relu(self.fc(x))
        x = self.relu1(self.fc1(x))
        x = self.fc2(x)
        return x

# Single image prediction
def predict(image_path):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Preprocessing
    transform = transforms.Compose([
        transforms.Grayscale(),
        transforms.Resize((32, 32)),
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,))
    ])

    # Load image
    image = Image.open(image_path)
    image = transform(image).unsqueeze(0).to(device)

    # Load model
    model = LeNet5(num_classes=65).to(device)
    model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=device))
    model.eval()

    # Inference
    with torch.no_grad():
        output = model(image)
        topk = torch.topk(output, k=TOP_K, dim=1)
        topk_indices = topk.indices.cpu().numpy()[0]
        topk_scores = topk.values.cpu().numpy()[0]

    print(f"\nTop-{TOP_K} Beam Predictions for {image_path}:")
    for i in range(TOP_K):
        print(f"Rank {i+1}: Beam {topk_indices[i]} (Score: {topk_scores[i]:.4f})")

    return topk_indices

## 6. VIBE-YOLOR — camera priming + radio-coordinate projection

`detect_car_bbox` (YOLOv11/COCO) + `estimate_beam_full_top3_from_box` (pinhole
projection onto the fixed `RX_BEAM_ANGLES` codebook, FOV=110°, W=960). This is
the paper's internal "VIBE-YOLOR" baseline (no closed loop).

In [ ]:
# -----------------------------------------------------------------------------
# VIBE-YOLOR (Section 6): camera priming + radio-coordinate projection, no loop.
#   detect_car_bbox()                  -> most-confident COCO 'car' box (norm.)
#   estimate_beam_full_top3_from_box() -> pinhole projection onto RX_BEAM_ANGLES
# -----------------------------------------------------------------------------
# === Load native YOLOv8x pretrained model (on COCO dataset) ===
yolo_model = YOLO('yolo11x.pt')  # Large, accurate model trained on 80 COCO classes

# === Check if CUDA is available and print model device info ===
if torch.cuda.is_available():
    device_str = "cuda"
    print("🟢 CUDA is available. YOLOR will run on GPU.")
else:
    device_str = "cpu"
    print("⚠️ CUDA not available. YOLOR will run on CPU.")

# === Car Detection Function ===
def detect_car_bbox(image_path, car_class_id=2):
    """
    Detects the most confident car in the image and returns:
    [x_center_norm, y_center_norm, width_norm, height_norm, class_id]
    """
    results = yolo_model.predict(source=image_path, device=device_str, verbose=False)
    detections = results[0].boxes.data.cpu().numpy()  # [x1, y1, x2, y2, conf, class]

    image_width, image_height = results[0].orig_shape[1], results[0].orig_shape[0]
    best_conf = -1
    best_bbox = None

    for det in detections:
        x1, y1, x2, y2, conf, cls = det
        if int(cls) != car_class_id:
            continue

        if conf > best_conf:
            x_center = (x1 + x2) / 2 / image_width
            y_center = (y1 + y2) / 2 / image_height
            width = (x2 - x1) / image_width
            height = (y2 - y1) / image_height
            best_bbox = [x_center, y_center, width, height, int(cls)]
            best_conf = conf

    return best_bbox



# === Constants ===
FOV_DEG = 110.0
IMAGE_WIDTH = 960
RX_BEAM_ANGLES = [
    0.0, -45.0, -43.5, -42.1, -40.6, -39.2, -37.7, -36.3, -34.8, -33.4, -31.9,
    -30.5, -29.0, -27.6, -26.1, -24.7, -23.2, -21.8, -20.3, -18.9, -17.4,
    -16.0, -14.5, -13.1, -11.6, -10.2, -8.7, -7.3, -5.8, -4.4, -2.9, -1.5, 0.0,
     1.5, 2.9, 4.4, 5.8, 7.3, 8.7, 10.2, 11.6, 13.1, 14.5, 16.0, 17.4,
     18.9, 20.3, 21.8, 23.2, 24.7, 26.1, 27.6, 29.0, 30.5, 31.9, 33.4,
     34.8, 36.3, 37.7, 39.2, 40.6, 42.1, 43.5, 45.0
]

def load_bbox(image_name):
    ann_path = os.path.join(bbox_base, image_name.replace(".jpg", ".txt"))
    if not os.path.exists(ann_path):
        return None
    try:
        with open(ann_path, 'r') as f:
            line = f.readline().strip()
            vals = list(map(float, line.split()))
            if len(vals) == 5:
                _, x_c, y_c, w, h = vals
                x_min = x_c - w / 2
                x_max = x_c + w / 2
                return [x_min, y_c, x_max, y_c]
            elif len(vals) == 4:
                return vals
    except:
        return None

def load_mmwave_power(image_name):
    try:
        rel_path = image_to_power_path.get(image_name, None)
        if rel_path is None or pd.isna(rel_path):
            return None
        rel_path = rel_path.lstrip("./")
        full_path = os.path.join(scenario_root, rel_path)
        if not os.path.exists(full_path):
            return None
        with open(full_path, 'r') as f:
            return np.array([float(line.strip()) for line in f if line.strip()])
    except:
        return None

def estimate_beam_full_top3_from_box(image_name, box):
    """
    Estimate left/center/right beam index from YOLO box, and return full metadata.
    """
    power_vals = load_mmwave_power(image_name)
    if box is None or power_vals is None or len(power_vals) != 64:
        return {
            "image_name": image_name,
            "true_beam_index": None,
            "true_beam_power": None,
            "est_theta_deg_left": None,
            "est_theta_deg_center": None,
            "est_theta_deg_right": None,
            "est_beam_index_left": None,
            "est_beam_index_center": None,
            "est_beam_index_right": None,
            "est_beam_angle_left": None,
            "est_beam_angle_center": None,
            "est_beam_angle_right": None,
            "estimated_beam_power_left": None,
            "estimated_beam_power_center": None,
            "estimated_beam_power_right": None,
            "beam_index_error_left": None,
            "beam_index_error_center": None,
            "beam_index_error_right": None,
            "hit_top1": None,
            "hit_top2": None,
            "hit_top3": None
        }

    x_c, y_c, w, h, cls = box

    positions = {
        "left": x_c - w / 2,
        "center": x_c,
        "right": x_c + w / 2
    }

    est_data = {}

    for key, x_norm in positions.items():
        u = x_norm * IMAGE_WIDTH
        delta_u = u - (IMAGE_WIDTH / 2)
        theta_deg = (delta_u / IMAGE_WIDTH) * FOV_DEG

        closest_angle = min(RX_BEAM_ANGLES, key=lambda x: abs(x - theta_deg))
        beam_idx = RX_BEAM_ANGLES.index(closest_angle) + 1  # 1-based
        beam_error = None
        beam_power = None
        if power_vals is not None:
            beam_error = abs(beam_idx - int(np.argmax(power_vals) + 1))
            beam_power = round(power_vals[beam_idx - 1], 2)

        est_data[f"est_theta_deg_{key}"] = round(theta_deg, 2)
        est_data[f"est_beam_index_{key}"] = beam_idx
        est_data[f"est_beam_angle_{key}"] = round(closest_angle, 2)
        est_data[f"estimated_beam_power_{key}"] = beam_power
        est_data[f"beam_index_error_{key}"] = beam_error

    true_idx = int(np.argmax(power_vals) + 1)
    true_power = round(power_vals[true_idx - 1], 2)

    top3 = [
        est_data["est_beam_index_center"],
        est_data["est_beam_index_left"],
        est_data["est_beam_index_right"]
    ]
    top1 = top3[:1]
    top2 = top3[:2]

    return {
        "image_name": image_name,
        "true_beam_index": true_idx,
        "true_beam_power": true_power,
        **est_data,
        "hit_top1": int(true_idx in top1),
        "hit_top2": int(true_idx in top2),
        "hit_top3": int(true_idx in top3)
    }

## 7. VIBE-MA — closed-loop neighbour search + offset tracking

`correct_beam_with_offset_tracking_single_v2`: outward ±1,±2,…,±delta search
around `est_beam + current_offset`; first beam clearing the Q-threshold wins
and its offset is carried to the next frame in the same pass.

In [ ]:
# -----------------------------------------------------------------------------
# VIBE-MA closed loop (Section 7) -- Algorithm 1 in the paper.
# Outward +/-1..+/-delta search around (est_beam+current_offset); first beam
# clearing the Q-threshold wins, offset carried to the next frame in the pass.
# -----------------------------------------------------------------------------
def correct_beam_with_offset_tracking_single_v2(
    est_beam: int,
    true_beam: int,
    power_vec: np.ndarray,
    delta: int,
    pwr_threshold: float,
    current_offset: int = 0,
    clear_offset: bool = False
):
    """
    Beam correction using neighbor search and offset tracking.

    Args:
        est_beam: Estimated beam index (1-based).
        true_beam: Ground-truth beam index (1-based).
        power_vec: Beam power vector (64 values).
        delta: Maximum offset range to search (δmax).
        pwr_threshold: Minimum acceptable power.
        current_offset: Offset to apply to estimated beam.
        clear_offset: If True, resets offset to 0.

    Returns:
        corrected_beam_index, corrected_power, beams_searched, corrected_error, new_offset
    """

    # Step 0: Validate input
    if power_vec is None or len(power_vec) < 64:
        return None, None, None, None, 0

    # Step 1: Check estimated beam directly
    est_power = power_vec[est_beam - 1]
    if est_power >= pwr_threshold:
        corrected_error = abs(pwr_threshold - est_beam)
        return est_beam, est_power, 1, corrected_error, 0  # Use offset = 0 when no search needed

    # Step 2: Begin neighbor search around offset-adjusted center
    offset = 0 if clear_offset else current_offset
    center = est_beam + offset
    bmin, bmax = 1, len(power_vec)

    best_beam = center
    best_power = -float('inf')
    beams_checked = 0

    # Search ±delta around center
    for delta_i in range(1, delta + 1):
        for direction in [-1, 1]:
            b = center + direction * delta_i
            if not (bmin <= b <= bmax):
                continue

            beams_checked += 1
            power = power_vec[b - 1]

            # If threshold is met, return with offset
            if power >= pwr_threshold:
                corrected_error = abs(pwr_threshold - b)
                new_offset = b - est_beam  # update offset only if threshold met
                return b, power, beams_checked, corrected_error, new_offset

            # Track best power (for fallback only)
            if power > best_power:
                best_power = power
                best_beam = b

    # Step 3: Fallback – no beam met threshold, so keep offset = 0
    corrected_error = abs(true_beam - best_beam)
    return best_beam, best_power, beams_checked, corrected_error, 0

## 8. Run all passes — the evaluation loop

For every image of every `passNN` folder it runs MNet+LeNet, VIBE-YOLOR and
VIBE-MA on the same image / ground-truth power vector and records one CSV row.
Outage is cumulative (`topk_outage = top1<Q and … and topk<Q`). VIBE-MA offset
state resets at the start of every pass. Output:
`MNet_LeNet_Model_Analysis/external_evaluation_results_Scenario7_PWR_<Q>.csv`.

---

### What is a *pass*?

DeepSense 6G delivers each scenario as long, time-ordered capture sequences.
To evaluate **beam tracking under motion**, the frames are regrouped into
`passNN/` folders (built by `sort_data_in_pass.py` from `scenarioX.csv`
using the dataset `seq_index`). One **pass = one continuous drive-by** of
the transmitter past the receiver: a contiguous, time-ordered run of frames
forming a single trajectory. The evaluation iterates pass-by-pass, and the
**VIBE-MA offset state is reset at the start of every pass** because each
pass is an independent trajectory (no offset should carry across passes).

In [ ]:
def process_image(idx, image_path, pass_name,
                  current_offset_lenet, current_offset_yolo):
    """
    Evaluate MobileNet+LeNet, VIBE-YOLOR and VIBE-MA on ONE image.

    Returns (row, current_offset_lenet, current_offset_yolo); row is the
    per-image result dict, or None if the image is skipped (no mmWave
    mapping / invalid power vector / true beam below the Q-threshold / no
    car detected). The two offsets are the VIBE-MA state carried to the
    next image in the same pass.
    """
    image_name = os.path.basename(image_path)
    mask_path = image_path.replace(".jpg", "_mask.png")

    print("------------------------------------------------------------------------------------------------")
    print("------------------------------------------------------------------------------------------------")
    
    print(f"🖼️ Processing image: {image_name}")

    # Validate mmWave power path
    if image_name not in image_to_power_path:
        print(f"⚠️ Skipping {image_name}: No mmWave mapping found.")
        return None, current_offset_lenet, current_offset_yolo

    mmwave_path = os.path.join(scenario_root, image_to_power_path[image_name].lstrip("./"))

    # Load mmWave power vector
    try:
        power_vec = load_mmwave_power(image_name)
    except Exception as e:
        print(f"❌ Error loading power vector for {image_name}: {e}")
        return None, current_offset_lenet, current_offset_yolo

    if power_vec is None or len(power_vec) != 64:
        print(f"❌ Skipping {image_name}: Invalid or missing power vector.")
        return None, current_offset_lenet, current_offset_yolo

    # === Compute true beam and threshold ===
    true_beam = int(np.argmax(power_vec) + 1)
    true_power = round(power_vec[true_beam - 1], 3)
    adaptive_threshold = round(np.quantile(power_vec, quantile_percentile), 4)
    max_power_all_beams = np.max(power_vec)

    print("\n📊 mmWave Vector Stats:")
    print(f"   - Max Beam Power: {max_power_all_beams:.3f}")
    print(f"   - True Beam: {true_beam}")
    print(f"   - True Beam Power: {true_power:.3f}")
    print(f"   - Quantile Threshold ({int(quantile_percentile*100)}%): {adaptive_threshold:.6f}")

    if true_power < adaptive_threshold:
        print(f"⚠️ Skipping {image_name}: True beam power ({true_power:.6f}) < Quantile threshold ({adaptive_threshold:.6f})")
        return None, current_offset_lenet, current_offset_yolo

    # ==========================================================
     # === PIPELINE: MobileNet+LeNet Prediction ===
    # ==========================================================
    try:
        print("------------------------------------------------------------------------------------------------")
        print(f"\n🧠 Running MobileNet + LeNet Beam Prediction for {image_name}")
    
        # Mask generation
        t1 = time.time()
        generate_mask_from_rgb(image_path=image_path, save_path=mask_path)
        mask_time = round(time.time() - t1, 4)
        print(f"🖼️ Mask generated in {mask_time} sec")
    
        # Beam prediction
        t2 = time.time()
        top_k_lenet = predict(mask_path)
        lenet_time = round(time.time() - t2, 4)
        print(f"🔮 LeNet beam prediction in {lenet_time} sec")
    
        # Top-k predictions and powers
        lenet_topk = top_k_lenet[:3]
        lenet_top1 = lenet_topk[0]
        lenet_top2 = lenet_topk[1] if lenet_topk[1] is not None else -1
        lenet_top3 = lenet_topk[2] if lenet_topk[2] is not None else -1
    
        power_top1 = round(power_vec[lenet_top1 - 1], 4) if 1 <= lenet_top1 <= 64 else 0
        power_top2 = round(power_vec[lenet_top2 - 1], 4) if 1 <= lenet_top2 <= 64 else 0
        power_top3 = round(power_vec[lenet_top3 - 1], 4) if 1 <= lenet_top3 <= 64 else 0

    
        # Cumulative outage logic
        lenet_top1_outage = power_top1 < adaptive_threshold
        lenet_top2_outage = lenet_top1_outage and (power_top2 < adaptive_threshold)
        lenet_top3_outage = lenet_top2_outage and (power_top3 < adaptive_threshold)
    
        print(f"\n📊 LeNet Top-k Results:")
        print(f"   - Top-1: Beam {lenet_top1}, Power = {power_top1:.3f}, Outage = {lenet_top1_outage}")
        print(f"   - Top-2: Beam {lenet_top2}, Power = {power_top2:.3f}, Outage = {lenet_top2_outage}")
        print(f"   - Top-3: Beam {lenet_top3}, Power = {power_top3:.3f}, Outage = {lenet_top3_outage}")

        print(f"🕒 Total MobileNet + LeNet time: {mask_time + lenet_time} sec")
    
    except Exception as e:
        logging.info(f"❌ LeNet pipeline failed: {e}")
        print(f"❌ LeNet pipeline failed: {e}")
        lenet_topk = [None, None, None]
        lenet_top1 = lenet_top2 = lenet_top3 = None
        power_top1 = power_top2 = power_top3 = 0
        lenet_top1_power_diff = lenet_top2_power_diff = lenet_top3_power_diff = 0
        lenet_top1_outage = lenet_top2_outage = lenet_top3_outage = False
        lenet_beam_index_error = lenet_beam_power_error = mask_time = lenet_time = 0

    # ==========================================================
     # === PIPELINE: VIBE-YOLOR → Vision-Based Beamforming ===
    # ==========================================================
    try:
        print("------------------------------------------------------------------------------------------------")
        t_yoloest = time.time()
        bbox = detect_car_bbox(image_path)
        detect_time = round(time.time() - t_yoloest, 4)
        print(f"📦 BBox detection time: {detect_time} sec")
        print(f"📦 Detected BBox: {bbox}")
    
        if bbox is None:
            print(f"❌ No car detected in {image_name}")
            return None, current_offset_lenet, current_offset_yolo
    
        t_beamest = time.time()
        yolor_result = estimate_beam_full_top3_from_box(image_name, bbox)
        yolo_beam_time = round(time.time() - t_beamest, 4)
    
        # Extract beam indices
        left = yolor_result["est_beam_index_left"]
        center = yolor_result["est_beam_index_center"]
        right = yolor_result["est_beam_index_right"]
    
        # Compute powers for each top K 
        power_left = round(power_vec[left - 1], 4) if 1 <= left <= 64 else 0
        power_center = round(power_vec[center - 1], 4) if 1 <= center <= 64 else 0
        power_right = round(power_vec[right - 1], 4) if 1 <= right <= 64 else 0

        # Compute outage flags
        yolo_top1_outage = power_center < adaptive_threshold
        yolo_top2_outage = yolo_top1_outage and (power_left < adaptive_threshold)
        yolo_top3_outage = yolo_top2_outage and (power_right < adaptive_threshold)

    
        print(f"\n🎯 YOLOR Predicted Beams:")
        print(f"   - Left: {left} (Power = {power_left:.3f}, Outage = {yolo_top2_outage})")
        print(f"   - Center: {center} (Power = {power_center:.3f}, Outage = {yolo_top1_outage})")
        print(f"   - Right: {right} (Power = {power_right:.3f}, Outage = {yolo_top3_outage})")
        
        print(f"🕒 Beam estimation time: {yolo_beam_time} sec")
        
    except Exception as e:
        print(f"❌ YOLOR beam estimation failed: {e}")
        return None, current_offset_lenet, current_offset_yolo

    # ==========================================================
     # === LeNet + MA Correction ===
    # ==========================================================
    try:
        print("------------------------------------------------------------------------------------------------")
        lenet_est = lenet_topk[0]
        est_power = power_vec[lenet_est - 1] if 1 <= lenet_est <= 64 else 0
        max_power = np.max(power_vec)
    
        print(f"\n LeNet Correction Logic")
        print(f"   - est_power (top1): {est_power:.6f}")
        print(f"   - max_power: {max_power:.6f}")
        print(f"   - threshold: {adaptive_threshold:.6f}")
        print(f"   - current_offset: {current_offset_lenet}")
    
        if max_power < adaptive_threshold:
            print("⚠️ Skipping correction due to low overall SNR")
    
        if max_power < adaptive_threshold or est_power >= adaptive_threshold:
            corrected_lenet = (lenet_est, est_power, 0, abs(true_beam - lenet_est), 0)
            lenet_corr_time = 0.0
            print(f"✅ No correction needed — top1 beam is sufficient")
        else:
            t6 = time.time()
            corrected_lenet = correct_beam_with_offset_tracking_single_v2(
                est_beam=lenet_top1,
                true_beam=true_beam,
                power_vec=power_vec,
                delta=delta,
                pwr_threshold=adaptive_threshold,
                current_offset=current_offset_lenet,
                clear_offset=(idx == 0)
            )
            lenet_corr_time = round(time.time() - t6, 6)
            new_beam = corrected_lenet[0]
            new_offset = new_beam - lenet_top1
            current_offset_lenet = new_offset
    
            print(f"🔧 Correction Applied:")
            print(f"   - Corrected Beam: {new_beam}")
            print(f"   - Corrected Power: {corrected_lenet[1]:.6f}")
            print(f"   - Correction Error (true vs corrected): {corrected_lenet[3]}")
            print(f"   - Beams Searched: {corrected_lenet[2]}")
            print(f"   - New Offset: {new_offset}")
            print(f"   - Correction Time: {lenet_corr_time:.4f} sec")
    
        lenet_corr_power_diff = round(adaptive_threshold - corrected_lenet[1], 4)
        lenet_corr_outage = corrected_lenet[1] < adaptive_threshold
    
        print(f"📉 Power Difference (adaptive - corrected): {lenet_corr_power_diff:.4f}")
        print(f"{' Outage' if lenet_corr_outage else '✅ Correction successful — power sufficient'}")
    
    except Exception as e:
        logging.info(f"❌ LeNet offset correction failed: {e}")
        corrected_lenet = (lenet_est, est_power, 0, abs(true_beam - lenet_est), 0)
        lenet_corr_time = 0
        lenet_corr_power_diff = 0
        lenet_corr_outage = True


    # ==========================================================
     # === VIBE-MA Correction ===
    # ==========================================================
    try:
        print("------------------------------------------------------------------------------------------------")
        est_power = power_vec[center - 1]
        max_power = np.max(power_vec)
    
        print(f"\n🛠️ Correction Logic")
        print(f"   - est_power (center): {est_power:.6f}")
        print(f"   - max_power: {max_power:.6f}")
        print(f"   - threshold: {adaptive_threshold:.6f}")
        print(f"   - current_offset: {current_offset_yolo}")
    
        if max_power < adaptive_threshold:
            print("⚠️ Skipping correction due to low overall SNR")
        
        if max_power < adaptive_threshold or est_power >= adaptive_threshold:
            corrected_yolo = (center, est_power, 0, abs(true_beam - center), 0)
            yolo_corr_time = 0.0
            print(f"✅ No correction needed — center beam is sufficient")
        else:
            yolo_corr = time.time()
            corrected_yolo = correct_beam_with_offset_tracking_single_v2(
                est_beam=center,
                true_beam=true_beam,
                power_vec=power_vec,
                delta=delta,
                pwr_threshold=adaptive_threshold,
                current_offset=current_offset_yolo,
                clear_offset=(idx == 0)
            )
            yolo_corr_time = round(time.time() - yolo_corr, 6)
            new_beam = corrected_yolo[0]
            new_offset = new_beam - center
            current_offset_yolo = new_offset
    
            print(f"🔧 Correction Applied:")
            print(f"   - Corrected Beam: {new_beam}")
            print(f"   - Corrected Power: {corrected_yolo[1]:.6f}")
            print(f"   - Correction Error (true vs corrected): {corrected_yolo[3]}")
            print(f"   - Beams Searched: {corrected_yolo[2]}")
            print(f"   - New Offset: {new_offset}")
            print(f"   - Correction Time: {yolo_corr_time:.4f} sec")
    
        yolo_corr_power_diff = round(adaptive_threshold - corrected_yolo[1], 4)
        yolor_corr_outage = corrected_yolo[1] < adaptive_threshold
    
        print(f"📉 Power Difference (adaptive - corrected): {yolo_corr_power_diff:.4f}")
        print(f"🚨 Correction Outage: {yolor_corr_outage}")
        

    except Exception as e:
        print(f"❌ YOLOR correction failed: {e}")
        corrected_yolo = (center, est_power, 0, abs(true_beam - center), 0)
        yolo_corr_time = 0
        yolo_corr_power_diff = 0
        yolor_corr_outage = True  # fallback: treat failure as outage

    
    print("##################################################################")
    
    # === Save result row ===
    row = {
        "image_name": image_name,
        "pass_name": pass_name,
        "true_beam_index": true_beam,
        "true_beam_power": true_power,
        "mmWave_Vector" : power_vec,
        "QuantileThreshold":adaptive_threshold,
        
        "lenet_top1": lenet_top1,
        "lenet_top2": lenet_top2,
        "lenet_top3": lenet_top3,
        "lenet_top1_power": round(power_top1, 6),
        "lenet_top2_power": round(power_top2, 6),
        "lenet_top3_power": round(power_top3, 6),
        "lenet_top1_outage": lenet_top1_outage,
        "lenet_top2_outage": lenet_top2_outage,
        "lenet_top3_outage": lenet_top3_outage,
   

        "yolor_top1_beam": center,
        "yolor_top1_power": power_center,
        "yolor_top1_outage": yolo_top1_outage,

        "lenet_corr_beam": corrected_lenet[0],
        "lenet_corr_error_index": corrected_lenet[3],
        "lenet_corr_beams_searched": corrected_lenet[2],
        "lenet_corr_power": round(corrected_lenet[1], 3),
        "lenet_corr_power_diff": lenet_corr_power_diff,
        "lenet_corr_outage": lenet_corr_outage,

        "yolor_corr_beam": corrected_yolo[0],
        "yolor_corr_error_index": corrected_yolo[3],
        "yolor_corr_beams_searched": corrected_yolo[2],
        "yolor_corr_power": round(corrected_yolo[1], 3),
        "yolor_corr_power_diff": yolo_corr_power_diff,
        "yolor_corr_outage": yolor_corr_outage,

        "timing_lenet_mask": mask_time,
        "timing_lenet_pred": lenet_time,
        "timing_lenet_correction": lenet_corr_time,
        "timing_yolo_detect": detect_time,
        "timing_yolo_beam": yolo_beam_time,
        "timing_yolo_correction": yolo_corr_time,
    }

    return row, current_offset_lenet, current_offset_yolo

In [ ]:
# =============================================================================
# Run the evaluation over every pass / image. process_image() (cell above) does
# all per-image work; here we enumerate passes, reset the VIBE-MA offsets per
# pass, thread them image-to-image, collect rows, and write the per-Q CSV.
# The dataset folder name has brackets ("DEV[95]") so globbing uses escape().
# =============================================================================
results = []

all_pass_folders = sorted([
    f for f in os.listdir(base_dir)
    if os.path.isdir(os.path.join(base_dir, f)) and f.startswith("pass")
])

for pass_name in all_pass_folders:
    print(f"\n[INFO] {pass_name}")
    current_offset_lenet = 0
    current_offset_yolo = 0

    pass_path = os.path.join(base_dir, pass_name)
    image_paths = sorted(glob(os.path.join(escape(pass_path), "*.jpg")))
    if not image_paths:
        print(f"[WARN] No images found in {pass_name}")
        continue
    print(f"[INFO] {len(image_paths)} images in {pass_name}")

    for idx, image_path in enumerate(image_paths):
        row, current_offset_lenet, current_offset_yolo = process_image(
            idx, image_path, pass_name,
            current_offset_lenet, current_offset_yolo,
        )
        if row is not None:
            results.append(row)

df_results = pd.DataFrame(results)
csv_path = os.path.join(
    OUTPUT_DIR, f"external_evaluation_results_Scenario7_PWR_{quantile_percentile}.csv"
)
df_results.to_csv(csv_path, index=False)
print(f"\n[INFO] Results saved to {csv_path}")


## 9. Roll up — outage / timing summary (Table II / Fig. 11)

In [ ]:
# Roll up the per-Q result CSVs into one outage / timing summary
# (feeds Table II / Fig. 11 of the paper).

files = {
    "80": os.path.join(OUTPUT_DIR, "external_evaluation_results_Scenario7_PWR_0.80.csv"),
    "90": os.path.join(OUTPUT_DIR, "external_evaluation_results_Scenario7_PWR_0.90.csv"),
    "95": os.path.join(OUTPUT_DIR, "external_evaluation_results_Scenario7_PWR_0.95.csv"),
}

outage_metrics = {
    "yolor_top1_outage": "YOLOR_Top1",
    "yolor_corr_outage": "YOLOR_Corr",
    "lenet_top1_outage": "LeNet_Top1",
    "lenet_top2_outage": "LeNet_Top2",
    "lenet_top3_outage": "LeNet_Top3",
    "lenet_corr_outage": "LeNet_Corr",
}
timing_metrics = {
    "YOLOR TopKTime": ["timing_yolo_detect", "timing_yolo_beam"],
    "YOLOR CorrTime": ["timing_yolo_detect", "timing_yolo_beam", "timing_yolo_correction"],
    "LeNet TopKTime": ["timing_lenet_mask", "timing_lenet_pred"],
    "LeNet CorrTime": ["timing_lenet_mask", "timing_lenet_pred", "timing_lenet_correction"],
}

summary = []
for snr_value, path in files.items():
    df = pd.read_csv(path)
    row = {}
    for col, label in outage_metrics.items():
        row[label] = round(100 * df[col].sum() / len(df), 2) if col in df.columns else "N/A"
    for method, cols in timing_metrics.items():
        row[method] = round(df[cols].sum(axis=1).mean(), 4) if all(c in df.columns for c in cols) else "N/A"
    summary.append(pd.Series(row, name=int(snr_value)))

final_df = pd.DataFrame(summary)
summary_path = os.path.join(OUTPUT_DIR, "combined_outage_timing_summary_Scenario7.csv")
final_df.to_csv(summary_path)
print(f"[INFO] Wrote summary -> {summary_path}")
print(final_df)